<a href="https://colab.research.google.com/github/Walnut235/olist-probabilistic-analysis/blob/main/notebooks/01_analisis_olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Probabilístico del E-Commerce Brasileño — Olist

## Machine Learning Probabilístico

**Dataset:** Brazilian E-Commerce Public Dataset by Olist

### Objetivo

Analizar el comportamiento del comercio electrónico de Olist mediante conceptos de probabilidad, estadística e información, con el fin de identificar patrones relevantes para la toma de decisiones.

### Conceptos a estudiar

1. Probabilidad condicional
2. Teorema de Bayes
3. Verosimilitud
4. Máxima verosimilitud (MLE)
5. Distribuciones paramétricas
6. Esperanza y varianza
7. Independencia y correlación
8. Prior y posterior
9. Entropía
10. Entropía cruzada
11. Divergencia KL

# Cargar datos

**Importar librerias**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

**Descarga del dataset de Olist**

In [4]:
!pip install -q kagglehub
# conectarse a kaggle
import kagglehub
import os

# Descargar el dataset de Olist
dataset_path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset descargado en:")
print(dataset_path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset descargado en:
/kaggle/input/brazilian-ecommerce


**Carga de las tablas**

In [6]:
# Diccionario para almacenar las tablas
data = {}
# Guardar las tablas en el diccionario
for file in files:
    file_path = os.path.join(dataset_path, file)
    table_name = file.replace(".csv", "")

    data[table_name] = pd.read_csv(file_path)

print("Tablas cargadas correctamente:")
for name, df in data.items():
    print(f"{name}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Tablas cargadas correctamente:
olist_customers_dataset: 99,441 filas × 5 columnas
olist_sellers_dataset: 3,095 filas × 4 columnas
olist_order_reviews_dataset: 99,224 filas × 7 columnas
olist_order_items_dataset: 112,650 filas × 7 columnas
olist_products_dataset: 32,951 filas × 9 columnas
olist_geolocation_dataset: 1,000,163 filas × 5 columnas
product_category_name_translation: 71 filas × 2 columnas
olist_orders_dataset: 99,441 filas × 8 columnas
olist_order_payments_dataset: 103,886 filas × 5 columnas


**Conocer las tablas**

In [10]:
for name, df in data.items(): # recorrer el diccionario
    print(f"\n{'=' * 60}")
    print(name) # nombre de la tabla
    print(f"{'=' * 60}")
    print("Columnas:")

    for column in df.columns: # mostrar las columnas del df
        print(f"  - {column}")



olist_customers_dataset
Columnas:
  - customer_id
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state

olist_sellers_dataset
Columnas:
  - seller_id
  - seller_zip_code_prefix
  - seller_city
  - seller_state

olist_order_reviews_dataset
Columnas:
  - review_id
  - order_id
  - review_score
  - review_comment_title
  - review_comment_message
  - review_creation_date
  - review_answer_timestamp

olist_order_items_dataset
Columnas:
  - order_id
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value

olist_products_dataset
Columnas:
  - product_id
  - product_category_name
  - product_name_lenght
  - product_description_lenght
  - product_photos_qty
  - product_weight_g
  - product_length_cm
  - product_height_cm
  - product_width_cm

olist_geolocation_dataset
Columnas:
  - geolocation_zip_code_prefix
  - geolocation_lat
  - geolocation_lng
  - geolocation_city
  - geolocation_state

product_category_name_tr

**Verificar relaciones (1:N, 1:1, M:N)**

In [8]:
# ¿Cada customer_id aparece una sola vez?
print("CUSTOMERS")
print("customer_id únicos:", data["olist_customers_dataset"]["customer_id"].nunique())
print("filas:", len(data["olist_customers_dataset"]))

print("\nORDERS")
print("order_id únicos:", data["olist_orders_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_orders_dataset"]))

print("\nPRODUCTS")
print("product_id únicos:", data["olist_products_dataset"]["product_id"].nunique())
print("filas:", len(data["olist_products_dataset"]))

print("\nSELLERS")
print("seller_id únicos:", data["olist_sellers_dataset"]["seller_id"].nunique())
print("filas:", len(data["olist_sellers_dataset"]))

print("\nREVIEWS")
print("order_id únicos:", data["olist_order_reviews_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_order_reviews_dataset"]))

print("\nORDER ITEMS")
print("order_id únicos:", data["olist_order_items_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_order_items_dataset"]))

CUSTOMERS
customer_id únicos: 99441
filas: 99441

ORDERS
order_id únicos: 99441
filas: 99441

PRODUCTS
product_id únicos: 32951
filas: 32951

SELLERS
seller_id únicos: 3095
filas: 3095

REVIEWS
order_id únicos: 98673
filas: 99224

ORDER ITEMS
order_id únicos: 98666
filas: 112650


**Union de las tablas**

In [22]:
# Guardar cada tabla en un df individual
orders = data["olist_orders_dataset"].copy()
customers = data["olist_customers_dataset"].copy()
items = data["olist_order_items_dataset"].copy()
products = data["olist_products_dataset"].copy()
sellers = data["olist_sellers_dataset"].copy()
reviews = data["olist_order_reviews_dataset"].copy()
payments = data["olist_order_payments_dataset"].copy()
geolocation = data["olist_geolocation_dataset"].copy()
category_translation = data["product_category_name_translation"].copy()

In [27]:
# ============================================
# 9. CONSTRUCCIÓN DEL DATAFRAME MAESTRO
# ============================================

df_master = orders.copy()

print("INICIAL")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

INICIAL
Filas: 99441
Pedidos únicos: 99441


In [28]:
df_master = df_master.merge(
    customers,
    on="customer_id",
    how="left",
    validate="one_to_one" #Relación 1:1
)

print("DESPUÉS DE CUSTOMERS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE CUSTOMERS
Filas: 99441
Pedidos únicos: 99441


In [29]:
df_master = df_master.merge(
    items,
    on="order_id",
    how="left",
    validate="one_to_many"
)

print("DESPUÉS DE ORDER_ITEMS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE ORDER_ITEMS
Filas: 113425
Pedidos únicos: 99441


In [36]:
df_master = df_master.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE PRODUCTS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE PRODUCTS
Filas: 113425
Pedidos únicos: 99441


In [38]:
df_master = df_master.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE SELLERS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE SELLERS
Filas: 113425
Pedidos únicos: 99441


In [39]:
df_master = df_master.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE CATEGORY TRANSLATION")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE CATEGORY TRANSLATION
Filas: 113425
Pedidos únicos: 99441


### Construcción y alcance del DataFrame maestro

Para la construcción del DataFrame maestro se realizaron uniones (`merge`) progresivas tomando como base la tabla `olist_orders_dataset`, donde cada registro representa un pedido. Primero se incorporó la información de los clientes mediante `customer_id` y posteriormente la información de los productos y vendedores a través de `order_items`, utilizando `product_id` y `seller_id`, respectivamente. Finalmente, se incorporó la traducción de las categorías mediante `product_category_name`. Se utilizaron uniones de tipo `left` para conservar los 99.441 pedidos originales, incluso cuando estos no contaban con información asociada en alguna de las tablas. Como resultado, el DataFrame pasó de 99.441 filas a 113.425 debido a que un mismo pedido puede contener varios productos. Las tablas de reseñas (`reviews`) y pagos (`payments`) no se incorporaron directamente al DataFrame maestro, ya que pueden contener múltiples registros asociados a un mismo pedido y podrían alterar su granularidad; estas serán agregadas posteriormente a nivel de `order_id` según las necesidades de cada análisis. La tabla de geolocalización tampoco se incorporó al DataFrame maestro debido a su gran cantidad de registros y a que un mismo código postal puede aparecer múltiples veces. Durante el proyecto, este DataFrame se utilizará como base para los análisis relacionados con pedidos, productos, categorías y vendedores, teniendo siempre en cuenta su granularidad de pedido-producto. Cuando un análisis requiera contar pedidos, clientes u otras entidades, se utilizarán identificadores únicos mediante `nunique()`, y cuando sea necesario obtener variables a nivel de pedido se realizarán las agregaciones correspondientes antes del análisis.

# limpieza de los datos

In [69]:
df_analysis.head() #Visulizar las 5 primeras filas

,order_id,customer_id,customer_unique_id,customer_state,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,seller_id,seller_state,product_category_name,product_weight_g
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,29.99,8.72,3504c0cb71d7fa48d967e0e4c94d59d9,SP,utilidades_domesticas,500.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,BA,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,118.70,22.76,289cdb325fb7e7f891c38608bf9e0962,SP,perfumaria,400.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,GO,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,159.90,19.22,4869f7a5dfa277a7dca6462dcf3b52b2,SP,automotivo,420.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,RN,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,45.00,27.20,66922902710d126a0e7d26b0e3805106,MG,pet_shop,450.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,SP,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,19.90,8.72,2c9e548be18521d1c43cde1c582c6de8,SP,papelaria,250.0


**Dejar solo las variables relevantes para el análisis**

In [44]:
variables_master = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_state",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "price",
    "freight_value",
    "seller_id",
    "seller_state",
    "product_category_name",
    "product_weight_g"
]

df_analysis = df_master[variables_master].copy()

print("Variables seleccionadas:")
print(df_analysis.columns.tolist())

print("\nDimensiones:")
print(df_analysis.shape)

Variables seleccionadas:
['order_id', 'customer_id', 'customer_unique_id', 'customer_state', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'price', 'freight_value', 'seller_id', 'seller_state', 'product_category_name', 'product_weight_g']

Dimensiones:
(113425, 14)


In [46]:
print("Columnas con valores faltantes:")
df_analysis.isna().sum() # Mirar que columnas tienen valores faltantes

Columnas con valores faltantes:


,0
order_id,0
customer_id,0
customer_unique_id,0
customer_state,0
order_status,0
order_purchase_timestamp,0
order_delivered_customer_date,3229
order_estimated_delivery_date,0
price,775
freight_value,775


In [53]:
df_analysis.dtypes #Mirar tipo de dato por columna

,0
order_id,object
customer_id,object
customer_unique_id,object
customer_state,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
price,float64
freight_value,float64


**Conversión de tipos de datos (fechas)**

In [50]:
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df_analysis[col] = pd.to_datetime(df_analysis[col], errors="coerce") #Convertir columnas de fechas a tipo datetime

print("Tipos de datos después de la conversión:")
print(df_analysis[date_cols].dtypes)


Tipos de datos después de la conversión:
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


**Diagnóstico de valores faltantes por `order_status`**

Antes de decidir qué hacer con cada columna con NaN, revisamos si los faltantes están relacionados con el estado del pedido (`order_status`). Esto nos permite distinguir entre un dato *realmente perdido* y un dato que *no existe porque el pedido nunca llegó a esa etapa* (por ejemplo, un pedido cancelado nunca tendrá fecha de entrega).

In [63]:
missing_delivery_orders = (
    df_analysis[df_analysis["order_delivered_customer_date"].isna()] # Filtrar los que no tienen fecha de entrega
    .drop_duplicates("order_id") #Trabajar a nivel de cada orden y no de item
)

print("Pedidos únicos sin fecha de entrega:", len(missing_delivery_orders))
print("\nDistribución de order_status:")
print(missing_delivery_orders["order_status"].value_counts())

Pedidos únicos sin fecha de entrega: 2965

Distribución de order_status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


 El pedido nunca fue entregado (shipped, canceled, unavailable, invoiced, processing) o, en 8 casos, quedó marcado `delivered` sin fecha registrada (inconsistencia del dato original)

In [64]:
missing_items_orders = (
    df_analysis[df_analysis["price"].isna()]
    .drop_duplicates("order_id")
)

print("Pedidos únicos sin ítem asociado:", len(missing_items_orders))
print("\nDistribución de order_status:")
print(missing_items_orders["order_status"].value_counts())

Pedidos únicos sin ítem asociado: 775

Distribución de order_status:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


 El pedido no tiene ningún ítem asociado en `olist_order_items_dataset`

In [68]:
missing_category = df_analysis[df_analysis["product_category_name"].isna()]# analisis de items, filtrar aquellos sin categoria
missing_weight = df_analysis[df_analysis["product_weight_g"].isna()]# analisis de items, filtrar aquellos sin peso

print("Filas sin categoría de producto:", len(missing_category))
print("Pedidos únicos afectados por falta de categoria:", missing_category["order_id"].nunique())# número de pedidos con algun item sin categoria
print("\nDistribución de order_status (categoría faltante):")
print(missing_category["order_status"].value_counts())

print("\nFilas sin peso de producto:", len(missing_weight))
print("Pedidos únicos afectados:", missing_weight["order_id"].nunique())# número de pedidos con algun item sin peso

Filas sin categoría de producto: 2378
Pedidos únicos afectados por falta de categoria: 2226

Distribución de order_status (categoría faltante):
order_status
delivered      1537
unavailable     603
canceled        178
shipped          28
invoiced         14
processing       13
created           5
Name: count, dtype: int64

Filas sin peso de producto: 793
Pedidos únicos afectados: 791


Inconsistencia conocida del dataset original de Olist, hay productos (`product_id`) sin categoría registrada en `olist_products_dataset`.Lo mismo pasa en el caso de los pesos de los productos.

**Tratamiento de valores faltantes**

No aplicamos `dropna()` global. Cada columna con NaN se trata según **por qué** falta el dato:

| Columna | NaN | Tratamiento |
|---|---|---|
| `order_delivered_customer_date` | 3.229 filas / 2.965 pedidos |  **Se conserva como NaN.** No se imputa una fecha inventada. Se usa para construir la variable derivada `retrasado`, que quedará en `NaN` quan no se pueda determinar. |
| `price`, `freight_value`, `seller_id`, `seller_state` | 775 filas | **Se conservan como NaN.** Estas filas se excluyen puntualmente solo en los análisis que dependen de precio/flete/vendedor (ej. esperanza y varianza del ticket, independencia pago–categoría), no se eliminan del DataFrame completo. |
| `product_category_name` | 2.378 filas | **Se imputa con la etiqueta `"sin_categoria"`** en lugar de NaN, para que estos productos sí cuenten como una categoría propia en los análisis de entropía e independencia (ignorarlos silenciosamente subestimaría la diversidad real del catálogo). |
| `product_weight_g` | 793 filas |  **Se conserva como NaN** y se excluye únicamente del análisis de distribución paramétrica del peso (no se imputa un peso, porque inventar un valor numérico sí distorsionaría el ajuste de la distribución). |

Ningún pedido se elimina del DataFrame por tener NaN en estas columnas — solo se excluye puntualmente de los análisis específicos que requieren esa variable.

In [70]:
df_clean = df_analysis.copy()

# Imputación puntual de product_category_name
df_clean["product_category_name"] = df_clean["product_category_name"].fillna("sin_categoria")#imputar "Sin categoria" cuando esta haga falta, a nivel de item

print("df_clean creado.")
print("Filas:", len(df_clean))
print("Columnas:", len(df_clean.columns))
print("\nValores faltantes restantes:")
print(df_clean.isna().sum())

df_clean creado.
Filas: 113425
Columnas: 14

Valores faltantes restantes:
order_id                            0
customer_id                         0
customer_unique_id                  0
customer_state                      0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    3229
order_estimated_delivery_date       0
price                             775
freight_value                     775
seller_id                         775
seller_state                      775
product_category_name               0
product_weight_g                  793
dtype: int64


**Variable derivada: `retrasado`**

Regla (ya definida): solo se clasifica un pedido como retrasado / no retrasado si fue efectivamente **entregado** y tiene fecha real de entrega. En cualquier otro caso, `retrasado = NaN` (no sabemos, no inventamos).

In [71]:
# Filas que tienen fecha de entrega pero NO están en estado delivered
inconsistencias = df_clean[
    df_clean["order_delivered_customer_date"].notna() &
    (df_clean["order_status"] != "delivered")
]

print("Filas con fecha de entrega pero estado diferente de 'delivered':", len(inconsistencias))

if len(inconsistencias) == 0:
    print("✓ Todos los registros con fecha de entrega pertenecen a pedidos 'delivered'.")
else:
    print("\nEstados encontrados:")
    print(inconsistencias["order_status"].value_counts())

Filas con fecha de entrega pero estado diferente de 'delivered': 7

Estados encontrados:
order_status
canceled    7
Name: count, dtype: int64


Estos registros fueron cancelados, pero aun así tienen una fecha de entrega registrada, lo cual representa una inconsistencia. A falta de información sobre su causa exacta, existen dos explicaciones plausibles: (1) un error de registro en la fecha, o (2) un fenómeno real, en el que el pedido llegó a entregarse (o inició su proceso logístico) y solo después fue cancelado en el sistema.

Como no es posible distinguir entre ambos escenarios con la información disponible, no se sobrescribe order_status ni se asume que la fecha implica una entrega real. Estos pedidos se conservan tal como están y, en consecuencia, quedan fuera de la clasificación de retrasado (NaN), igual que el resto de pedidos no entregados. Esta inconsistencia se documenta como limitación conocida del dataset, junto con los 8 casos delivered sin fecha.

In [72]:
fue_entregado = (
    (df_clean["order_status"] == "delivered") &
    (df_clean["order_delivered_customer_date"].notna())#asi se excluyen los registros que fueron entregados pero no tienen fecha de entrega registrada
)

df_clean["retrasado"] = np.where(
    fue_entregado, #¿pertenece a fue entregado?
    (df_clean["order_delivered_customer_date"] > df_clean["order_estimated_delivery_date"]).astype(int),
    np.nan #de lo contrario NaN
)

print("Distribución de 'retrasado' (incluye NaN = no clasificable):")
print(df_clean["retrasado"].value_counts(dropna=False))

print("\nPedidos únicos clasificables:", df_clean.loc[df_clean["retrasado"].notna(), "order_id"].nunique())

Distribución de 'retrasado' (incluye NaN = no clasificable):
retrasado
0.0    101475
1.0      8714
NaN      3236
Name: count, dtype: int64

Pedidos únicos clasificables: 96470


**Validaciones de consistencia**

In [75]:
print("1) Precios y fletes negativos:")
print("   price < 0:", (df_clean["price"] < 0).sum())
print("   freight_value < 0:", (df_clean["freight_value"] < 0).sum())

print("\n2) Consistencia cronológica (compra <= entrega, cuando hay fecha real):")
inconsistentes = df_clean[
    df_clean["order_delivered_customer_date"].notna() &
    (df_clean["order_delivered_customer_date"] < df_clean["order_purchase_timestamp"])
]
print("   Pedidos con fecha de entrega ANTES de la compra:", inconsistentes["order_id"].nunique())

print("\n3) Rango de fechas del dataset:")
print("   Compra: ", df_clean["order_purchase_timestamp"].min(), "→", df_clean["order_purchase_timestamp"].max())

1) Precios y fletes negativos:
   price < 0: 0
   freight_value < 0: 0

2) Consistencia cronológica (compra <= entrega, cuando hay fecha real):
   Pedidos con fecha de entrega ANTES de la compra: 0

3) Rango de fechas del dataset:
   Compra:  2016-09-04 21:15:19 → 2018-10-17 17:30:18
